# Step 03 Viz Demo

Renders `fixtures/apartment_minimal.json` via `proto3.viz.render` and shows the resulting SVG inline.

## Prerequisites

1. `pip install -e ".[dev]"` (one-time, from repo root)
2. `nbstripout --install` (one-time, activates the `*.ipynb filter=nbstripout` registered in `.gitattributes`)
3. The next code cell finds the repo root automatically by walking up until `pyproject.toml` is found, so this notebook works whether you launch jupyter from repo root or open it in VSCode (which sets cwd to the notebook's directory).

## Output convention (S03-D13)

Generated artifacts go to `outputs/notebooks/step03_viz_demo/<run_id>/` where `run_id` is an ISO timestamp. Each re-run creates a new folder; older runs are kept for comparison. `outputs/` is gitignored (D014).

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing pyproject.toml is found."""
    p = start.resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"pyproject.toml not found from {start} upward")


ROOT = _find_repo_root(Path.cwd())
print("repo root:", ROOT)

In [ ]:
from proto3.schema.input import BuildingInput
from proto3.schema.serialize import from_json

fixture_path = ROOT / "fixtures" / "apartment_minimal.json"
building = from_json(BuildingInput, fixture_path.read_text())
print("target_type:", building.target_type)
print("floors:", len(building.floors))
print("footprint vertices:", len(building.floors[0].footprint))
print("program_request:", building.program_request)

In [ ]:
from datetime import datetime

from proto3.viz import render

run_id = datetime.now().strftime("%Y%m%dT%H%M%S")
out_dir = ROOT / "outputs" / "notebooks" / "step03_viz_demo" / run_id
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "minimal.svg"
render(building, out_path=str(out_path))
print("rendered to:", out_path.relative_to(ROOT))

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(out_path)))

---

## Notes

- Re-running creates a new `run_id` folder under `outputs/notebooks/step03_viz_demo/`. Old runs are kept.
- The SVG declares all 12 `<g class="layer-NN-name">` groups in the [D013](../000_Architecture_Decisions.md) order. Layers without data in Step 03 (anchors, regions, graph-edges, role-scores, spine, slots, seeds, grown, doors, failure) emit empty groups. Later Steps fill them.
- Coordinate convention: input mm with math-y up; SVG flips to display-y down (S03-D6).